# ENCODE Search

This notebook demonstrates `run_encode_search`, which searches the ENCODE portal by facet filters (assay, target, biosample, output type, assembly, file format) and optional free text. It returns processed files or experiments with their accessions, metadata and direct download URLs. Here we find the default IDR thresholded peak file for MAFK TF ChIP-seq in HepG2, then list CTCF ChIP-seq experiments with a free-text search.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("encode")
display_overview("encode")
display_docs_section("encode", "Background")

# ENCODE

The [ENCODE portal](https://www.encodeproject.org) hosts the experiments and uniformly processed files of the Encyclopedia of DNA Elements, including ChIP-seq, ATAC-seq, DNase-seq, RNA-seq and Hi-C across human and mouse biosamples. The `encode-search` tool runs faceted searches of the portal by assay, target, biosample, output type and assembly, and returns each file's or experiment's accession, metadata and direct download URL.

The ENCODE Consortium maps functional elements in the human and mouse genomes: transcription factor binding, histone modifications, chromatin accessibility, transcription and 3D genome contacts. The ENCODE Data Coordination Center at Stanford runs the portal, which is described in [*New developments on the Encyclopedia of DNA Elements (ENCODE) data portal*](https://doi.org/10.1093/nar/gkz1062) by Luo et al., published in *Nucleic Acids Research*.

Data is organized as **experiments** (`ENCSR...` accessions), each with a biosample, an assay and, for ChIP-seq, a target. Each experiment holds **files** (`ENCFF...` accessions): raw reads, and the outputs of the uniform processing pipelines such as alignments, signal tracks and peak calls on a given genome assembly. For each output type and assembly, the portal flags one file as the **preferred default**, for example the replicated IDR thresholded peaks of a TF ChIP-seq experiment.

The tool queries the portal's [REST API](https://www.encodeproject.org/help/rest-api/), whose search endpoint accepts the same facet filters as the portal's search pages and returns JSON.

## Available tools

In [2]:
display_available_tools("encode")

- **`run_encode_search()`** — Search the ENCODE portal for experiments and processed files (peaks, signal, alignments) by assay, target, biosample and assembly; returns accessions, metadata and download URLs

### `run_encode_search`

`run_encode_search` runs one portal search per query. A query is a mapping of ENCODE facet filters, whose values must match the portal's terms exactly, plus an optional `search_term`. Each hit carries the file or experiment accession, its parent experiment, assay, target, biosample, output type, assembly, size, MD5 checksum, preferred-default flag and, for files, the download URL.

In [3]:
from proto_tools import (
    EncodeSearchConfig,
    EncodeSearchInput,
    run_encode_search,
)

In [4]:
# Display docs
display_api_reference("encode", "input", "run_encode_search")

# Default GRCh38 IDR thresholded peaks for MAFK TF ChIP-seq in HepG2
inputs = EncodeSearchInput(
    queries={
        "assay_title": "TF ChIP-seq",
        "target.label": "MAFK",
        "biosample_ontology.term_name": "HepG2",
        "output_type": "IDR thresholded peaks",
        "assembly": "GRCh38",
        "file_format": "bed",
    }
)

**Input** — `EncodeSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>queries</code> | <code>list[EncodeSearchQuery]</code> | required | ENCODE searches; a bare dict is accepted |

In [5]:
# Display config docs
display_api_reference("encode", "config", "run_encode_search")

# Keep only the portal's preferred default file | see docs above for all fields
config = EncodeSearchConfig(default_only=True)

**Config** — `EncodeSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>item_type</code> | <code>Literal['File', 'Experiment']</code> | <code>'File'</code> | Search files or experiments |
| <code>released_only</code> | <code>bool</code> | <code>True</code> | Restrict to released items |
| <code>default_only</code> | <code>bool</code> | <code>False</code> | Restrict files to the portal's preferred default files |
| <code>max_results</code> | <code>int</code> | <code>25</code> | Maximum hits returned per query |

In [6]:
# Run the search
output = run_encode_search(inputs, config)

In [7]:
# Display output docs
display_api_reference("encode", "output", "run_encode_search")

result = output.results[0]
print(f"{result.total_count} matching file(s)")
for hit in result.hits:
    print(f"{hit.accession} from {hit.dataset_accession}: {hit.output_type}, {hit.assembly}, {hit.file_type}")
    print(f"  replicates {hit.biological_replicates}, {hit.file_size} bytes, md5 {hit.md5sum}")
    print(f"  {hit.download_url}")

**Output** — `EncodeSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[EncodeSearchResult]</code> | <code>[]</code> | Results in input order |
| <code>source_url</code> | <code>str</code> | required | ENCODE search endpoint used |

**`EncodeSearchResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query</code> | <code>EncodeSearchQuery</code> | required | Query as given |
| <code>request_url</code> | <code>str</code> | required | Full search URL sent |
| <code>total_count</code> | <code>int</code> | required | Total matching items |
| <code>hits</code> | <code>list[EncodeSearchHit]</code> | <code>[]</code> | Hits in portal order |

**`EncodeSearchQuery`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>filters</code> | <code>dict[str, str]</code> | <code>{}</code> | ENCODE facet filters, e.g. {'target.label': 'CTCF'} |
| <code>search_term</code> | <code>str &#124; None</code> | <code>None</code> | Free-text search term |

**`EncodeSearchHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | File or experiment accession |
| <code>dataset_accession</code> | <code>str &#124; None</code> | <code>None</code> | Parent experiment |
| <code>assay_title</code> | <code>str &#124; None</code> | <code>None</code> | Assay title |
| <code>target</code> | <code>str &#124; None</code> | <code>None</code> | Assay target label |
| <code>biosample</code> | <code>str &#124; None</code> | <code>None</code> | Biosample term name |
| <code>output_type</code> | <code>str &#124; None</code> | <code>None</code> | File output type |
| <code>assembly</code> | <code>str &#124; None</code> | <code>None</code> | Genome assembly |
| <code>file_format</code> | <code>str &#124; None</code> | <code>None</code> | File format |
| <code>file_type</code> | <code>str &#124; None</code> | <code>None</code> | Detailed file type |
| <code>file_size</code> | <code>int &#124; None</code> | <code>None</code> | File size in bytes |
| <code>md5sum</code> | <code>str &#124; None</code> | <code>None</code> | File MD5 checksum |
| <code>biological_replicates</code> | <code>list[int]</code> | <code>[]</code> | Source biological replicates |
| <code>preferred_default</code> | <code>bool &#124; None</code> | <code>None</code> | Portal default file for its type |
| <code>status</code> | <code>str &#124; None</code> | <code>None</code> | Portal status |
| <code>lab</code> | <code>str &#124; None</code> | <code>None</code> | Submitting lab |
| <code>date_released</code> | <code>str &#124; None</code> | <code>None</code> | Release date |
| <code>date_created</code> | <code>str &#124; None</code> | <code>None</code> | Creation date |
| <code>description</code> | <code>str &#124; None</code> | <code>None</code> | Experiment description |
| <code>download_url</code> | <code>str &#124; None</code> | <code>None</code> | Direct file download URL |
| <code>portal_url</code> | <code>str</code> | required | Portal page for the item |

1 matching file(s)
ENCFF654RRF from ENCSR000EEB: IDR thresholded peaks, GRCh38, bed narrowPeak
  replicates [1, 2], 1238443 bytes, md5 2dde26453e92db5c7101410349de9a9c
  https://www.encodeproject.org/files/ENCFF654RRF/@@download/ENCFF654RRF.bed.gz


#### Searching Experiments with Free Text

With `item_type="Experiment"` the tool searches experiments instead of files. Free text in `search_term` is useful when the exact facet terms are unknown, and can be combined with filters. Here we list released CTCF TF ChIP-seq experiments in K562.

In [8]:
exp_output = run_encode_search(
    EncodeSearchInput(queries={"search_term": "CTCF K562", "filters": {"assay_title": "TF ChIP-seq"}}),
    EncodeSearchConfig(item_type="Experiment", max_results=5),
)

exp_result = exp_output.results[0]
print(f"{exp_result.total_count} matching experiment(s); top {len(exp_result.hits)}:")
for hit in exp_result.hits:
    print(f"{hit.accession} | {hit.target} | {hit.biosample} | {hit.lab} | released {hit.date_released}")

6 matching experiment(s); top 5:
ENCSR000AKO | CTCF | K562 | Bradley Bernstein, Broad | released 2011-02-10
ENCSR000DMA | CTCF | K562 | Vishwanath Iyer, UTA | released 2012-08-27
ENCSR000BPJ | CTCF | K562 | Richard Myers, HAIB | released 2012-02-29
ENCSR000DWE | CTCF | K562 | John Stamatoyannopoulos, UW | released 2011-03-17
ENCSR000BNK | CTCFL | K562 | Richard Myers, HAIB | released 2011-07-18


## Export Results

Results can be saved as JSON (the full output) or CSV (one row per hit) for downstream use.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("encode_mafk_hepg2_idr_peaks", export_path=output_dir, file_format="csv")
print(f"Exported to {output_dir / 'encode_mafk_hepg2_idr_peaks.csv'}")

Exported to example_output/encode_mafk_hepg2_idr_peaks.csv
